# 4 — Monte Carlo: answering questions by playing them out

Some questions have an answer you can write down. Most interesting ones do not.

A Monte Carlo simulation replaces the algebra with repetition: **play the game a
hundred thousand times and count.** It is the most broadly useful numerical
technique there is, and the whole method fits in four lines.

We will use a deck of cards, because the answers can be checked exactly — which
means you find out whether your simulation is right, rather than believing it.

By the end you will be able to:

* generate random numbers **reproducibly** (the part people get wrong);
* estimate a probability by simulation, and say how accurate the estimate is;
* work out how many simulations you actually need;
* run a hundred thousand trials without a Python loop;
* simulate whole *paths*, not just averages, and read the distribution.

**Assumed:** notebooks 1–3 — loops, functions, NumPy arrays, a bit of plotting.

**Not in here:** no regression. This notebook generates data rather than fitting
models to it.

In [ ]:
# Run me first. This makes workbook.py importable whatever folder Jupyter
# started in, then pulls in the three helpers you will use all the way through.
import sys
from pathlib import Path

for candidate in [Path.cwd(), Path.cwd() / 'Learn_To_Code', *Path.cwd().parents]:
    if (candidate / 'workbook.py').exists():
        sys.path.insert(0, str(candidate))
        break

from workbook import check, hint, todo, ensure_data, DATA_DIR

ensure_data()   # builds the workbook CSVs on your Desktop the first time only

print('Ready. Data lives in:', DATA_DIR)

In [ ]:
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.axisbelow': True})

## 1. Random numbers you can reproduce

Start every simulation with a **generator**, created once, with a seed:

```python
rng = np.random.default_rng(42)
```

Two reasons this specific line matters.

**It is reproducible.** Same seed, same numbers, forever — on your machine and on
your marker's. A simulation you cannot reproduce is not a result, it is an
anecdote.

**It is local.** The older functions — `np.random.seed(42)` then `np.random.rand()`
— share one hidden generator for the whole process. Any library you import can
draw from it and shift your results. `default_rng` hands you your own; nobody else
can touch it. If you see `np.random.seed` in old code, it still works, but do not
write new code that way.

In [ ]:
rng = np.random.default_rng(42)

print('uniform [0,1)  ', rng.random(3).round(4))
print('integers 0..51 ', rng.integers(0, 52, size=5))
print('normal         ', rng.normal(0.0, 1.0, 3).round(4))
print('pick 3 of 10   ', rng.choice(10, size=3, replace=False))
print('shuffled 0..9  ', rng.permutation(10))

In [ ]:
# Reproducibility, demonstrated.
a = np.random.default_rng(7).random(4)
b = np.random.default_rng(7).random(4)
c = np.random.default_rng(8).random(4)

print('seed 7 :', a.round(4))
print('seed 7 :', b.round(4), '  identical')
print('seed 8 :', c.round(4), '  different')

⚠️ **The mistake that ruins simulations.** Create the generator **once**, outside
the loop. Creating it inside — `rng = np.random.default_rng(42)` on every
iteration — resets the stream every time, so all ten thousand of your "random"
trials are the same trial. The code runs, the answer is confidently wrong, and
nothing warns you.

### Your turn

In [ ]:
# Show that two generators with the same seed produce the same numbers.
# Draw at least 3 values from each.

first  = todo()
second = todo()

print(first, second)
check('4.1', first, second)

## 2. A deck of cards

Two ways to represent a card, and the choice matters more than it looks.

**For humans:** a pair like `('A', '♠')`. Readable, easy to print.

**For simulation:** a plain integer 0–51, where

```python
rank = card % 13      # 0..12   ->  2,3,...,10,J,Q,K,A
suit = card // 13     # 0..3    ->  spades, hearts, diamonds, clubs
```

The integer version is the one that will let NumPy deal a hundred thousand hands
at once in section 6. **Picking the representation that makes the computation easy
is most of the work in a simulation** — the cleverness usually lives there rather
than in the algorithm.

In [ ]:
RANKS = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']
SUITS = ['s', 'h', 'd', 'c']


def card_name(card):
    """Turn a card index 0-51 into something readable."""
    return RANKS[card % 13] + SUITS[card // 13]


print([card_name(c) for c in [0, 12, 13, 51]])
print('ranks of 0, 13, 26, 39:', [c % 13 for c in [0, 13, 26, 39]],
      '  <- same rank, four suits')

In [ ]:
rng = np.random.default_rng(2024)

hand = rng.choice(52, size=5, replace=False)     # replace=False: no card twice
print('indices:', hand)
print('hand   :', [card_name(c) for c in hand])
print('ranks  :', hand % 13)

`replace=False` is doing real work there. Leave it out and you are dealing from an
infinite deck where the same card can come up twice — a different game, with
different answers.

### Your turn

In [ ]:
# Build a full deck as a list of 52 distinct cards, in whatever human-readable
# form you like — ('A', 's'), 'As', (12, 0) all count. Two loops or one
# comprehension over RANKS and SUITS.

deck = todo()

print(len(deck), 'cards |', deck[:5] if len(deck) else '')
check('4.2', deck)

## 3. One trial, written as a function

Every Monte Carlo has the same skeleton:

```python
1. write a function that plays the game ONCE and returns the outcome
2. call it many times
3. average the outcomes
```

Step 1 is where the thinking goes. If one trial is right, the rest is arithmetic.

**Our question:** deal five cards from a shuffled deck — how often does the hand
contain at least one pair?

One trial means: deal five cards, look at the five ranks, and decide whether any
rank appears twice. The neat way to ask that is to count the *distinct* ranks: five
cards with no pair have five different ranks, so anything fewer than five means a
repeat.

In [ ]:
def deal(rng, n=5):
    """Deal n cards from a fresh shuffled deck."""
    return rng.choice(52, size=n, replace=False)


rng = np.random.default_rng(1)
for _ in range(5):
    h = deal(rng)
    ranks = h % 13
    print(f'{[card_name(c) for c in h]}  ranks {np.sort(ranks)}  '
          f'distinct {len(np.unique(ranks))}')

### Your turn

In [ ]:
# Write has_pair(cards): True if any two of the five cards share a rank.
# `cards` arrives as an array of 5 integers between 0 and 51.
# np.unique() gives you the distinct values of an array.

def has_pair(cards):
    pass    # replace this line


print(has_pair(np.array([0, 13, 5, 7, 9])), ' <- two aces, should be True')
print(has_pair(np.array([0, 1, 2, 3, 4])), ' <- all different, should be False')
check('4.3', has_pair)

## 4. The Monte Carlo loop

Now repeat and count. The estimate is just the fraction of trials that came out
True — which is what an average of ones and zeros is.

In [ ]:
def estimate_pair_probability(n_trials, seed=0):
    rng = np.random.default_rng(seed)      # ONCE, outside the loop
    hits = 0
    for _ in range(n_trials):
        if len(np.unique(deal(rng) % 13)) < 5:
            hits += 1
    return hits / n_trials


for n in [100, 1_000, 10_000]:
    t0 = time.perf_counter()
    est = estimate_pair_probability(n)
    print(f'{n:>7,} trials -> {est:.4f}   ({time.perf_counter() - t0:.2f}s)')

### Is it right?

This is why we chose cards. The exact answer is combinatorics you can check:

$$P(\text{no pair}) = \frac{\binom{13}{5}\,4^5}{\binom{52}{5}}$$

— choose 5 of the 13 ranks, pick a suit for each, over all possible hands.
`math.comb` does the binomial coefficients.

In [ ]:
p_no_pair = math.comb(13, 5) * 4 ** 5 / math.comb(52, 5)
P_EXACT = 1 - p_no_pair

print(f'exact P(at least one pair) = {P_EXACT:.6f}')

**Always do this when you can.** Test your simulation on a case with a known
answer *before* you point it at the question you actually care about. A simulation
that is quietly wrong looks exactly like one that is right.

### Your turn

In [ ]:
# Estimate P(at least one pair) with at least 20,000 trials, using your own
# has_pair. Then compare it with the exact answer.

n_trials = 20_000
rng = np.random.default_rng(123)

estimate = todo()

print(f'estimate {estimate}   exact {P_EXACT:.4f}')
check('4.4', estimate, n_trials)

## 5. Watching it converge

Plot the running estimate against the number of trials. It wanders early and
settles later — that settling is the law of large numbers, and seeing it is worth
more than reading about it.

In [ ]:
rng = np.random.default_rng(99)
outcomes = np.array([len(np.unique(deal(rng) % 13)) < 5 for _ in range(20_000)])

running = np.cumsum(outcomes) / np.arange(1, len(outcomes) + 1)
n_axis = np.arange(1, len(outcomes) + 1)
band = 1.96 * np.sqrt(P_EXACT * (1 - P_EXACT) / n_axis)

fig, ax = plt.subplots()
ax.plot(n_axis, running, linewidth=1, label='running estimate')
ax.axhline(P_EXACT, color='crimson', linestyle='--', label=f'exact {P_EXACT:.4f}')
ax.fill_between(n_axis, P_EXACT - band, P_EXACT + band,
                color='crimson', alpha=0.12, label='95% band')

ax.set_xscale('log')
ax.set_xlim(10, len(outcomes))
ax.set_ylim(0.35, 0.65)
ax.set_xlabel('Number of trials (log scale)')
ax.set_ylabel('Estimated probability')
ax.set_title('Converging on the right answer, slowly')
ax.legend()
plt.show()

## 6. How wrong is your estimate? (the section that matters)

An estimate without an error bar is not an answer. For a probability estimated
from `n` independent trials:

$$\mathrm{se} = \sqrt{\frac{p(1-p)}{n}}$$

and the 95% interval is roughly `estimate ± 2·se`.

Look at the shape of it: **the error falls with the square root of n.** A hundred
times more simulation buys you ten times more accuracy — which is why "just run
more" stops being a strategy quite quickly.

Turn it around and it becomes a planning tool. If you need a standard error of
`target`, you need

$$n = \frac{p(1-p)}{\mathrm{target}^2}$$

trials. Work that out *before* you start the run, not after.

In [ ]:
p = P_EXACT
for n in [1_000, 10_000, 100_000, 1_000_000]:
    se = math.sqrt(p * (1 - p) / n)
    print(f'n = {n:>9,}   se = {se:.5f}   95% CI +/- {1.96 * se:.4f}')

print()
for target in [0.01, 0.001, 0.0005]:
    need = p * (1 - p) / target ** 2
    print(f'to reach se = {target:<7} you need {need:>12,.0f} trials')

### Rare events are expensive

The formula has a nasty consequence. The *relative* error of a rare event blows
up: to pin down something that happens 0.2% of the time, you need a hair-splitting
absolute standard error, and therefore an enormous number of trials.

A flush — five cards of the same suit — has exact probability
`4·C(13,5)/C(52,5)`.

In [ ]:
p_flush = 4 * math.comb(13, 5) / math.comb(52, 5)
print(f'P(flush) = {p_flush:.5f}  (about 1 hand in {1 / p_flush:.0f})')

for n in [10_000, 100_000, 1_000_000]:
    se = math.sqrt(p_flush * (1 - p_flush) / n)
    print(f'n = {n:>9,}   se = {se:.6f}   relative error {se / p_flush:.1%}')

Ten thousand trials — which felt like plenty for the pair question — gives you the
flush probability to within about 30%. That is not an answer, and nothing in the
output would have told you so if you had not computed the standard error.

This is the single most useful habit in this notebook: **report the standard error
with every Monte Carlo number you produce.**

### Your turn

In [ ]:
# With n = 20,000 trials and p = P_EXACT:
#   se       : the standard error of the estimate
#   n_needed : how many trials to get the standard error down to 0.0005

se       = todo()
n_needed = todo()

print(f'se {se}   need {n_needed} trials')
check('4.5', se, n_needed)

## 7. Doing it without a loop

Notebook 1 made the case for vectorisation on arithmetic. It applies just as much
to simulation, and the trick for dealing cards is genuinely elegant.

To shuffle one deck you could sort 52 random numbers and use the order. To shuffle
**a hundred thousand decks at once**, do exactly that in two dimensions:

```python
rng.random((n_sims, 52)).argsort(axis=1)[:, :5]
```

* `rng.random((n_sims, 52))` — a random number for every card of every deck;
* `.argsort(axis=1)` — the order of each row, which is a shuffle of 0–51;
* `[:, :5]` — take the first five. Five distinct cards, guaranteed.

Then detect pairs across all the hands at once: sort each row of ranks, and a
repeated rank shows up as a zero difference between neighbours.

In [ ]:
def pair_probability_vectorised(n_sims, seed=0):
    rng = np.random.default_rng(seed)
    hands = rng.random((n_sims, 52)).argsort(axis=1)[:, :5]
    ranks = np.sort(hands % 13, axis=1)
    has_repeat = (np.diff(ranks, axis=1) == 0).any(axis=1)
    return has_repeat.mean(), hands


def pair_probability_partition(n_sims, seed=0):
    """Same thing, but we only need the 5 smallest — no need to sort all 52."""
    rng = np.random.default_rng(seed)
    hands = np.argpartition(rng.random((n_sims, 52)), 5, axis=1)[:, :5]
    ranks = np.sort(hands % 13, axis=1)
    return (np.diff(ranks, axis=1) == 0).any(axis=1).mean()


def timed(label, fn, n):
    t0 = time.perf_counter()
    est = fn(n)
    dt = time.perf_counter() - t0
    print(f'{label:<26}{n:>9,} sims  {dt:6.2f}s  {dt / n * 1e6:6.2f} us/sim  '
          f'-> {est:.4f}')
    return dt / n


t_loop = timed('python loop', estimate_pair_probability, 20_000)
t_sort = timed('vectorised, argsort', lambda n: pair_probability_vectorised(n)[0], 200_000)
t_part = timed('vectorised, argpartition', pair_probability_partition, 200_000)

print(f'\nexact {P_EXACT:.4f}')
print(f'argsort is {t_loop / t_sort:.1f}x faster per simulation than the loop')
print(f'argpartition is {t_loop / t_part:.1f}x faster')

### Read that speed-up carefully

Three or four times faster, then roughly twice that again (your exact numbers will
differ — timings depend on the machine). That is useful, and it is a long way
short of the fifty-fold difference notebook 1 got from vectorising arithmetic. The
reason is worth understanding, because it tells you when vectorising is worth your
time.

**Vectorising removes Python's per-iteration overhead. It does not remove the
work.** Here the work per trial is real — 52 random numbers, then a sort — so
taking the interpreter out of the loop only saves the smaller part of the cost.
When the per-trial work is trivial (multiply a number by two), the interpreter
*is* the cost, and removing it wins enormously.

The second speed-up is a different kind of improvement: `argpartition` only
separates the 5 smallest values from the rest instead of ordering all 52. Same
answer, less work. Once vectorising has taken the interpreter out of the picture,
the next gains come from asking for less — which is an algorithm question, not a
NumPy one.

**The cost of all this is memory.** `rng.random((1_000_000, 52))` is 416 MB before
you have done anything with it. When a simulation outgrows memory, run it in
batches of a hundred thousand and accumulate the counts: the loop comes back, but
around blocks rather than around trials.

### Your turn

In [ ]:
# Deal at least 50,000 five-card hands with no Python loop, then estimate
# P(at least one pair) from them.

rng = np.random.default_rng(555)

hands    = todo()   # shape (n_sims, 5), distinct cards in each row
estimate = todo()

print(hands.shape, estimate)
check('4.6', hands, estimate)

## 8. From a probability to a game

Probabilities are the warm-up. What you usually want is a **payoff**.

**High Card.** One shuffled deck. You get a card, the house gets a card. Higher
rank wins \$1. **Ties go to the house.**

That one sentence about ties is the entire business model. Compute it exactly:
after your card is dealt, 51 cards remain and 3 of them match your rank, so

* P(tie) = 3/51
* P(you win) = P(not tie)/2 = 24/51
* P(house wins) = 27/51

Expected profit per \$1 bet = 24/51 − 27/51 = **−3/51 = −5.88%**. The house edge
is exactly the tie probability. Nothing else about the game is unfair.

Simulate it and see the number appear.

In [ ]:
def play_high_card(n_hands, seed=0):
    """Deal 2 cards from one deck, n_hands times. Returns your profit per hand."""
    rng = np.random.default_rng(seed)
    two = rng.random((n_hands, 52)).argsort(axis=1)[:, :2] % 13   # your rank, house rank
    return np.where(two[:, 0] > two[:, 1], 1.0, -1.0)             # ties lose


profit = play_high_card(200_000, seed=11)

edge = -profit.mean()
se = profit.std() / math.sqrt(len(profit))

print(f'simulated house edge {edge:.4f}  +/- {1.96 * se:.4f}')
print(f'exact 3/51           {3 / 51:.4f}')

Note what the last two lines do: the standard error tells you the simulation and
the algebra agree to within the noise. Without it, "0.0587 versus 0.0588" is a
coincidence you are choosing to believe.

### Your turn

In [ ]:
# Estimate the house edge from at least 200,000 simulated hands.
# Report it as a POSITIVE number: the share of each $1 bet the house keeps.

house_edge = todo()

print(f'{house_edge:.4f}  vs exact {3 / 51:.4f}')
check('4.7', house_edge)

## 9. Paths, not averages

"You lose 5.88 cents per dollar" describes the average player. Nobody is the
average player. What actually happens to a person is a **path**, and the spread of
paths is usually the thing you care about.

Give 5,000 players \$20 each and let them bet \$1 a hand, 200 times.
`np.cumsum(..., axis=1)` turns per-hand profits into a running bankroll.

In [ ]:
N_PLAYERS, N_HANDS, START = 5_000, 200, 20.0

rng = np.random.default_rng(4)
deals = rng.random((N_PLAYERS * N_HANDS, 52)).argsort(axis=1)[:, :2] % 13
per_hand = np.where(deals[:, 0] > deals[:, 1], 1.0, -1.0).reshape(N_PLAYERS, N_HANDS)

bankroll = START + np.cumsum(per_hand, axis=1)

print('bankroll array shape:', bankroll.shape, ' (one row per player)')
print(f'average final bankroll {bankroll[:, -1].mean():.2f}  '
      f'(expected {START - N_HANDS * 3 / 51:.2f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for i in range(40):                                  # 40 individual players
    axes[0].plot(bankroll[i], linewidth=0.8, alpha=0.6)
axes[0].plot(bankroll.mean(axis=0), color='black', linewidth=2.5, label='average')
axes[0].axhline(0, color='crimson', linewidth=1.2, label='broke')
axes[0].set_xlabel('Hand')
axes[0].set_ylabel('Bankroll ($)')
axes[0].set_title('40 players. The average describes none of them')
axes[0].legend()

# 200 bets of +/-1 always leave an even bankroll, so line the bins up with the
# values that can actually occur. Mismatched bins on discrete data produce a
# comb of empty gaps that looks like structure and is not.
final = bankroll[:, -1]
axes[1].hist(final, bins=np.arange(final.min() - 1, final.max() + 3, 2),
             color='#0173B2', alpha=0.85)
axes[1].axvline(START, color='0.3', linestyle='--', label='started here')
axes[1].axvline(final.mean(), color='crimson', label='mean')
axes[1].set_xlabel('Final bankroll ($)')
axes[1].set_ylabel('Number of players')
axes[1].set_title('Where 5,000 players ended up')
axes[1].legend()

fig.tight_layout()
plt.show()

In [ ]:
final = bankroll[:, -1]
ever_broke = (bankroll.min(axis=1) <= 0).mean()

print(f'ended ahead            {(final > START).mean():.1%}')
print(f'went broke at any point {ever_broke:.1%}')
print(f'worst final bankroll   {final.min():.0f}')
print(f'best final bankroll    {final.max():.0f}')

Almost one player in five finishes ahead of where they started — and every one of
them will tell you the game is beatable. Meanwhile two in five go broke at some
point. The simulation gives you the average *and* the spread, which is exactly
what a single expected-value calculation hides.

Note the difference between *ending* below zero and *ever* touching zero. Paths
let you ask path-dependent questions — drawdowns, stop-losses, margin calls, ruin
— and those are usually the questions that matter.

### Your turn

In [ ]:
# Simulate at least 2,000 players over at least 50 hands.
#   paths  : 2-D array, one row per player, running bankroll along each row
#   p_ruin : the share of players whose bankroll ever hit 0 or below

rng = np.random.default_rng(77)

paths  = todo()
p_ruin = todo()

print(f'ruin probability {p_ruin}')
check('4.8', paths, p_ruin)

## 10. Beyond cards: continuous randomness

Cards are discrete. Most quantities you will simulate are not, and the method does
not change at all — only the line that draws the randomness.

```python
rng.normal(mean, sd, size)      rng.lognormal(...)      rng.uniform(lo, hi, size)
rng.standard_t(df, size)        rng.binomial(n, p, size)
```

Simulate 10,000 ten-year investment paths: 120 monthly returns each, \$100 to
start, drawn from a normal with a mean of 0.7% and a standard deviation of 4.5% a
month.

The row-and-column layout is the same as the bankroll: **rows are simulations,
columns are time.** Compounding runs along `axis=1`.

In [ ]:
N_SIMS, N_MONTHS, START = 10_000, 120, 100.0

rng = np.random.default_rng(2025)
monthly = rng.normal(0.007, 0.045, size=(N_SIMS, N_MONTHS))

paths = START * np.cumprod(1 + monthly, axis=1)
terminal = paths[:, -1]

print(f'mean    {terminal.mean():8,.1f}')
print(f'median  {np.median(terminal):8,.1f}')
for q in [5, 25, 50, 75, 95]:
    print(f'{q:>3}th percentile {np.percentile(terminal, q):8,.1f}')

**The mean is well above the median.** That is not a bug and it is not an artefact
of the seed — it is what compounding does. Returns add up in logs, so terminal
wealth is skewed right: a handful of very good paths pull the average up past what
a typical path achieves.

The practical consequence: **the average outcome of a compounding process is not
the typical outcome.** If someone quotes you an expected terminal value, ask for
the median and the 5th percentile too. The simulation gives you all three for
free; the closed-form expectation gives you the one that flatters.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for i in range(60):
    axes[0].plot(paths[i], linewidth=0.7, alpha=0.5)
axes[0].plot(np.median(paths, axis=0), color='black', linewidth=2.5, label='median path')
axes[0].set_yscale('log')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Value ($, log scale)')
axes[0].set_title('60 of 10,000 paths')
axes[0].legend()

axes[1].hist(terminal, bins=60, color='#0173B2', alpha=0.85)
axes[1].axvline(np.median(terminal), color='black', linewidth=2, label='median')
axes[1].axvline(terminal.mean(), color='crimson', linewidth=2, label='mean')
axes[1].axvline(np.percentile(terminal, 5), color='0.4', linestyle='--',
                label='5th percentile')
axes[1].set_xlabel('Terminal value ($)')
axes[1].set_ylabel('Simulations')
axes[1].set_title('Right-skewed, because compounding')
axes[1].legend()

fig.tight_layout()
plt.show()

### Your turn

In [ ]:
# Simulate at least 5,000 ten-year paths of $1,000 with monthly returns drawn
# from normal(mean=0.005, sd=0.04), then report the median and the 5th
# percentile of the terminal value.

rng = np.random.default_rng(31415)

terminal = todo()   # 1-D array of terminal values, one per simulation
median   = todo()
p05      = todo()

print(f'median {median:,.0f}   5th percentile {p05:,.0f}')
check('4.9', terminal, median, p05)

## 11. Optional: getting more from the same number of draws

Since accuracy only improves with `sqrt(n)`, the other way to get a better answer
is to make each draw work harder. The simplest trick is **antithetic variates**:
for every random draw `z`, also use `−z`. The pairs are negatively correlated, so
their errors partly cancel.

In [ ]:
def terminal_mean(n_pairs, antithetic, seed=0):
    rng = np.random.default_rng(seed)
    if antithetic:
        z = rng.normal(size=(n_pairs, N_MONTHS))
        z = np.vstack([z, -z])                    # each path and its mirror image
    else:
        z = rng.normal(size=(2 * n_pairs, N_MONTHS))
    return (100.0 * np.cumprod(1 + 0.007 + 0.045 * z, axis=1))[:, -1].mean()


plain = [terminal_mean(2_500, False, seed=s) for s in range(40)]
anti  = [terminal_mean(2_500, True,  seed=s) for s in range(40)]

print(f'plain      : spread across 40 runs = {np.std(plain):.2f}')
print(f'antithetic : spread across 40 runs = {np.std(anti):.2f}')
print(f'-> {np.std(plain) / np.std(anti):.1f}x less variable, same 5,000 paths')

Same cost, a tighter answer. There is a family of these — control variates,
importance sampling, stratification — and they are how production simulations stay
affordable. Antithetic variates are the one to know first because they are three
lines and almost always help for symmetric distributions.

## 12. The mistakes that produce confident nonsense

| Mistake | What you see | Fix |
| ------- | ------------ | --- |
| generator created inside the loop | suspiciously smooth results, zero variance | create `rng` once, outside |
| no seed at all | different answer every run, nothing reproducible | always seed |
| same seed for two "independent" experiments | fake correlation between them | different seeds, or one generator used throughout |
| too few trials | an answer that moves when you rerun it | compute the standard error |
| no standard error reported | you cannot tell agreement from coincidence | report `p ± 2·se` |
| rare event, ordinary sample size | huge relative error, invisible | size the run from the target se |
| sampling with replacement by accident | probabilities slightly off, no error raised | `replace=False` when it is one deck |
| validating on nothing | a wrong simulation that looks fine | test against a case with an exact answer |

## 13. Where you are

| | |
| --- | --- |
| **randomness** | `default_rng`, seeds, reproducibility, and why the global functions are a trap |
| **the method** | one trial as a function, repeat, average |
| **accuracy** | `se = sqrt(p(1-p)/n)`, the `sqrt(n)` law, sizing a run in advance |
| **speed** | vectorising a whole simulation, and the memory it costs |
| **payoffs** | expected value, and the house edge of a game |
| **distributions** | paths, quantiles, ruin, and why the mean is not the typical outcome |
| **efficiency** | antithetic variates |

### Where this goes next

Everything here generalises with almost no new code:

* replace the card deal with **resampling your own historical returns** and you
  have a bootstrap — a way to get standard errors for statistics that have no
  formula;
* replace the payoff with an option's, and you have priced it by simulation;
* feed the residuals from notebook 3 back through a model and you can simulate
  what your regression implies rather than only what it fitted.

The skeleton never changes: **write one trial, repeat it, average, and report the
standard error.**

---

That is the workbook. Notebook 1 gave you the language, notebook 2 the picture,
notebook 3 the estimate, and notebook 4 the experiment. The project README shows
how to point all four at live data.

In [ ]:
from workbook import exercises
for nb in (1, 2, 3, 4):
    print(f'Notebook {nb}: {", ".join(exercises(nb))}')